<a href="https://colab.research.google.com/github/ZahiaYanes/medmcqa-lora-expert-fusion/blob/main/notebooks/03_merging__evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — Model Merging and Comparison

Notebook 02 showed that domain specialization works, but with an unexpected twist:
there is **no cross-domain cost**. Every expert improved every domain relative to
the base model — consistent positive transfer rather than a trade-off.

This reframes the merging question. The original motivation ("each expert is
narrow, merge them for broad coverage") does not hold. We therefore compare four
approaches:

| Approach | Principle |
|---|---|
| Base | Frozen Qwen2.5-1.5B-Instruct |
| Individual experts | Three adapters, each trained on one domain |
| **Merged** | Weight-space average of the three adapters |
| **Jointly trained** | A single adapter trained on all three domains pooled |

The last one is the serious competitor: if positive transfer is what drives the
gains, training on pooled data may already capture it — and merging would only be
an approximation of that.

## Methodological Note

Notebook 02 sampled evaluation questions sequentially (`subject_data[:n]`), which
turned out to be biased: the validation set is not randomly ordered. The
Pharmacology expert scored 56.0% on the first 150 questions but 60.9% on all 243.

This notebook samples randomly with a fixed seed, so all figures below are
comparable to each other but **not directly comparable** to the tables in
notebook 02. All four approaches (base, individual experts, merged, jointly
trained) are re-evaluated here under identical conditions.

In [14]:
!pip install -U datasets transformers peft accelerate bitsandbytes torchao -q

In [15]:
# Re-execution control
TRAIN_JOINT   = False    #  set to False once the adapter is saved
RUN_FULL_EVAL = False   #  final evaluation of all four approaches

In [16]:
import torch, os, gc, random
import pandas as pd

assert torch.cuda.is_available(), "No GPU detected! Runtime > Change runtime type > T4"
print("Running on:", torch.cuda.get_device_name(0))

Running on: Tesla T4


In [17]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_ROOT = "/content/drive/MyDrive/medmcqa-experts"
SUBJECTS = ["Pharmacology", "Pathology", "Surgery"]

print("Available adapters:", os.listdir(SAVE_ROOT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Available adapters: ['pharmacology', 'pathology', 'surgery', 'joint', 'merged']


In [18]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

dataset = load_dataset("openlifescienceai/medmcqa")

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Ready.")

Ready.


In [19]:
LETTER_MAP = {0: "A", 1: "B", 2: "C", 3: "D"}
MAX_LENGTH = 256

def build_prompt(example):
    return f"""Answer the following medical question by responding with only the letter of the correct option (A, B, C, or D).

Question: {example['question']}
A) {example['opa']}
B) {example['opb']}
C) {example['opc']}
D) {example['opd']}

Answer:"""

def tokenize_with_masking(example):
    messages = [{"role": "user", "content": build_prompt(example)}]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    answer_text = LETTER_MAP[example["cop"]] + tokenizer.eos_token

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(answer_text, add_special_tokens=False)["input_ids"]

    input_ids = prompt_ids + answer_ids
    labels = [-100] * len(prompt_ids) + answer_ids   # mask the prompt

    if len(input_ids) > MAX_LENGTH:
        overflow = len(input_ids) - MAX_LENGTH
        input_ids, labels = input_ids[overflow:], labels[overflow:]

    return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids), "labels": labels}

def free_memory(*objects):
    for obj in objects:
        del obj
    gc.collect()
    torch.cuda.empty_cache()

## Jointly Trained Adapter

A single LoRA adapter trained on the three domains pooled together (45,504
examples). Same hyperparameters as the individual experts, so the comparison
isolates the effect of the training data composition.

Training data is shuffled so batches mix domains, rather than seeing them
sequentially.

In [20]:
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, padding=True, label_pad_token_id=-100
)

joint_path = f"{SAVE_ROOT}/joint"

if TRAIN_JOINT:
    joint_data = dataset["train"].filter(lambda x: x["subject_name"] in SUBJECTS)
    joint_data = joint_data.shuffle(seed=42)   # mix domains within batches
    print(f"Joint training examples: {len(joint_data)}")

    joint_tokenized = joint_data.map(
        tokenize_with_masking, remove_columns=joint_data.column_names
    )

    base = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.float16, device_map="auto"
    )
    joint_model = get_peft_model(base, lora_config)
    joint_model.config.use_cache = False

    args = TrainingArguments(
        output_dir="outputs/checkpoints/joint",
        num_train_epochs=1,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=100,
        save_strategy="no",
        report_to="none",
    )

    Trainer(
        model=joint_model,
        args=args,
        train_dataset=joint_tokenized,
        data_collator=data_collator,
    ).train()

    joint_model.save_pretrained(joint_path)
    print("Saved:", joint_path)
    free_memory(joint_model, base)
else:
    print("Skipped — joint adapter already trained.")

Skipped — joint adapter already trained.


## Weight-Space Merging

The three adapters are combined by averaging their weights — the "model soup"
approach. Each contributes equally (1/3).

This is only valid because all three share the same rank and target the same
modules: their weight matrices have identical shapes and can be averaged
element-wise.

Unlike the jointly trained adapter, merging requires **no additional training** —
it is a purely arithmetic operation on already-trained weights.

In [21]:
from peft import PeftModel

merged_path = f"{SAVE_ROOT}/merged"

base = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map="auto"
)

# Load the three adapters side by side under distinct names
merged_model = PeftModel.from_pretrained(
    base, f"{SAVE_ROOT}/pharmacology", adapter_name="pharmacology"
)
merged_model.load_adapter(f"{SAVE_ROOT}/pathology", adapter_name="pathology")
merged_model.load_adapter(f"{SAVE_ROOT}/surgery", adapter_name="surgery")

# Linear average of the three weight sets
merged_model.add_weighted_adapter(
    adapters=["pharmacology", "pathology", "surgery"],
    weights=[1/3, 1/3, 1/3],
    adapter_name="merged",
    combination_type="linear",
)
merged_model.set_adapter("merged")

merged_model.save_pretrained(merged_path, selected_adapters=["merged"])
print("Saved:", merged_path)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/medmcqa-experts/merged


## Faster Evaluation: Logit Scoring

Notebook 02 evaluated by generating tokens and parsing the output letter. That
costs five forward passes per question and requires string parsing.

Here we instead read the logits at the final position and compare the scores of
the four option tokens directly, taking the argmax. This is:

- **Faster** — one forward pass per question, and questions can be batched
- **Cleaner** — no parsing, no unparsable outputs by construction

Because all six models are evaluated identically, comparisons remain valid.
Absolute numbers will differ slightly from notebook 02.

In [22]:
LETTERS = ["A", "B", "C", "D"]
LETTER_TOKEN_IDS = [tokenizer.encode(l, add_special_tokens=False)[0] for l in LETTERS]

def evaluate_logits(eval_model, subject_name, n_samples=150, batch_size=8, seed=42):
    """Score the four option tokens directly from the logits at the last position."""
    eval_model.eval()

    data = [ex for ex in dataset["validation"] if ex["subject_name"] == subject_name]
    rng = random.Random(seed)   # fixed seed: same questions for every model
    data = rng.sample(data, min(n_samples, len(data)))

    tokenizer.padding_side = "left"   # so the last position is the answer slot

    correct = 0
    for i in range(0, len(data), batch_size):
        batch = data[i:i + batch_size]
        texts = [
            tokenizer.apply_chat_template(
                [{"role": "user", "content": build_prompt(ex)}],
                tokenize=False, add_generation_prompt=True
            )
            for ex in batch
        ]
        inputs = tokenizer(
            texts, return_tensors="pt", padding=True, add_special_tokens=False
        ).to(eval_model.device)

        with torch.no_grad():
            logits = eval_model(**inputs).logits[:, -1, :]

        preds = logits[:, LETTER_TOKEN_IDS].argmax(dim=-1).tolist()
        correct += sum(p == ex["cop"] for p, ex in zip(preds, batch))

    tokenizer.padding_side = "right"
    return correct / len(data), correct, len(data)

In [23]:
import json

N_EVAL = 150
results = {}

def eval_all_domains(m, label):
    row = {}
    for subj in SUBJECTS:
        acc, c, t = evaluate_logits(m, subj, N_EVAL)
        row[subj] = acc * 100
        print(f"  {label:12s} {subj:14s}: {acc:.2%} ({c}/{t})")
    return row

if RUN_FULL_EVAL:
    # Base + merged share one loaded model
    print("=== BASE ===")
    with merged_model.disable_adapter():
        results["Base"] = eval_all_domains(merged_model, "Base")

    print("=== MERGED ===")
    results["Merged"] = eval_all_domains(merged_model, "Merged")
    free_memory(merged_model, base)

    # Individual experts + joint adapter
    for name in ["pharmacology", "pathology", "surgery", "joint"]:
        b = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=torch.float16, device_map="auto"
        )
        m = PeftModel.from_pretrained(b, f"{SAVE_ROOT}/{name}")
        print(f"=== {name.upper()} ===")
        results[name.capitalize()] = eval_all_domains(m, name[:10])
        free_memory(m, b)

    with open(f"{SAVE_ROOT}/final_results.json", "w") as f:
        json.dump(results, f)

    df = pd.DataFrame(results).T
    df["Mean"] = df.mean(axis=1)
    print(df.round(2).to_string())
else:
    print("Skipped.")

=== BASE ===
  Base         Pharmacology  : 51.33% (77/150)
  Base         Pathology     : 50.00% (75/150)
  Base         Surgery       : 43.33% (65/150)
=== MERGED ===
  Merged       Pharmacology  : 60.00% (90/150)
  Merged       Pathology     : 51.33% (77/150)
  Merged       Surgery       : 41.33% (62/150)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

=== PHARMACOLOGY ===
  pharmacolo   Pharmacology  : 60.00% (90/150)
  pharmacolo   Pathology     : 52.67% (79/150)
  pharmacolo   Surgery       : 44.00% (66/150)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

=== PATHOLOGY ===
  pathology    Pharmacology  : 56.00% (84/150)
  pathology    Pathology     : 56.67% (85/150)
  pathology    Surgery       : 48.67% (73/150)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

=== SURGERY ===
  surgery      Pharmacology  : 54.00% (81/150)
  surgery      Pathology     : 52.67% (79/150)
  surgery      Surgery       : 52.00% (78/150)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

=== JOINT ===
  joint        Pharmacology  : 62.67% (94/150)
  joint        Pathology     : 50.67% (76/150)
  joint        Surgery       : 49.33% (74/150)
              Pharmacology  Pathology  Surgery   Mean
Base                 51.33      50.00    43.33  48.22
Merged               60.00      51.33    41.33  50.89
Pharmacology         60.00      52.67    44.00  52.22
Pathology            56.00      56.67    48.67  53.78
Surgery              54.00      52.67    52.00  52.89
Joint                62.67      50.67    49.33  54.22


In [24]:
def build_merge(combination_type, name, rank=None):
    """Merge the three adapters using a given combination strategy."""
    b = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.float16, device_map="auto"
    )
    m = PeftModel.from_pretrained(b, f"{SAVE_ROOT}/pharmacology", adapter_name="pharmacology")
    m.load_adapter(f"{SAVE_ROOT}/pathology", adapter_name="pathology")
    m.load_adapter(f"{SAVE_ROOT}/surgery", adapter_name="surgery")

    kwargs = {"svd_rank": rank} if rank else {}
    m.add_weighted_adapter(
        adapters=["pharmacology", "pathology", "surgery"],
        weights=[1/3, 1/3, 1/3],
        adapter_name=name,
        combination_type=combination_type,
        **kwargs,
    )
    m.set_adapter(name)
    return m, b

In [25]:
if RUN_FULL_EVAL:
  # "cat" is exact: concatenating factors reproduces the true sum of deltas
  m, b = build_merge("cat", "merged_cat")
  print("=== MERGED (cat) ===")
  results["Merged_cat"] = eval_all_domains(m, "cat")
  free_memory(m, b)

  # "svd" approximates the true sum while keeping the original rank
  m, b = build_merge("svd", "merged_svd", rank=16)
  print("=== MERGED (svd) ===")
  results["Merged_svd"] = eval_all_domains(m, "svd")
  free_memory(m, b)

  df = pd.DataFrame(results).T
  df["Mean"] = df.mean(axis=1)
  print(df.round(2).to_string())
else:
    print("Skipped — results preserved above.")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

=== MERGED (cat) ===
  cat          Pharmacology  : 58.67% (88/150)
  cat          Pathology     : 56.00% (84/150)
  cat          Surgery       : 49.33% (74/150)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

=== MERGED (svd) ===
  svd          Pharmacology  : 58.00% (87/150)
  svd          Pathology     : 56.67% (85/150)
  svd          Surgery       : 50.00% (75/150)
              Pharmacology  Pathology  Surgery   Mean
Base                 51.33      50.00    43.33  48.22
Merged               60.00      51.33    41.33  50.89
Pharmacology         60.00      52.67    44.00  52.22
Pathology            56.00      56.67    48.67  53.78
Surgery              54.00      52.67    52.00  52.89
Joint                62.67      50.67    49.33  54.22
Merged_cat           58.67      56.00    49.33  54.67
Merged_svd           58.00      56.67    50.00  54.89


### Finding 1: naive linear merging is harmful for LoRA

Averaging adapters with `combination_type="linear"` scored **below the base model
on Surgery** (41.33 vs 43.33) and ranked last among all adapters.

The cause is structural. A LoRA adapter is a *product* of two matrices
($\Delta W = B \cdot A$), not a single weight matrix. Averaging the factors
separately gives:

$$
B_{\text{merged}} \cdot A_{\text{merged}}
= \underbrace{w_1 B_1 A_1 + w_2 B_2 A_2}_{\text{intended}}
+ \underbrace{w_1 B_2 A_1 + w_2 B_1 A_2}_{\text{cross-terms}}
$$

The cross-terms pair one expert's $B$ with another's $A$. They encode nothing that
was ever learned — they are noise injected into the model.

`combination_type="cat"` concatenates the factors instead, which reproduces the
intended sum exactly:

$$
A_{\text{merged}} = \begin{bmatrix} A_1 \\ A_2 \\ A_3 \end{bmatrix}, \quad
B_{\text{merged}} = \begin{bmatrix} w_1 B_1 & w_2 B_2 & w_3 B_3 \end{bmatrix}
\quad \Longrightarrow \quad
B_{\text{merged}} \cdot A_{\text{merged}} = \sum_i w_i B_i A_i
$$

Switching strategies recovers **+3.8 points of mean accuracy** and **+8 points on
Surgery**, with no retraining.

### Finding 2: correct merging matches joint training at negligible cost

| Approach | Mean accuracy | GPU time |
|---|---|---|
| Joint training (45,504 examples) | 54.22 | 54 min |
| Merging (cat) | 54.67 | ~5 s |
| Merging (svd, rank 16) | 54.89 | ~5 s |

These three are statistically indistinguishable (95% margin of error ≈ ±4.6 points
on 450 questions). The practical implication is that **merging reaches joint-training
performance without any additional training** — provided the merge is done correctly.

SVD merging additionally compresses back to the original rank, so the merged adapter
stays at ~17 MB, versus ~50 MB for `cat` (rank 48).

### Caveats

- With n=150 per domain, per-cell margin of error is roughly ±8 points. Only the
  linear-merge degradation and the base-to-adapter gains clearly exceed noise.
- Absolute numbers differ from notebook 02 because two things changed at once
  (random sampling instead of sequential, and logit scoring instead of generation).
  The two notebooks are internally consistent but not directly comparable.
- Domains in MedMCQA overlap conceptually (a pharmacology question may involve
  pathology). This likely explains the positive transfer observed in notebook 02
  and limits how "separable" these experts really are.
- Single seed, single epoch, one model size. Conclusions about merging strategies
  would need multiple seeds to be robust.

## Summary

Four approaches were compared on 150 randomly sampled validation questions per
domain (450 total), all evaluated identically via logit scoring.

| Model | Pharmacology | Pathology | Surgery | Mean |
|---|---|---|---|---|
| Base | 51.33 | 50.00 | 43.33 | 48.22 |
| Merged (linear) | 60.00 | 51.33 | 41.33 | 50.89 |
| Pharmacology expert | 60.00 | 52.67 | 44.00 | 52.22 |
| Surgery expert | 54.00 | 52.67 | 52.00 | 52.89 |
| Pathology expert | 56.00 | 56.67 | 48.67 | 53.78 |
| Jointly trained | **62.67** | 50.67 | 49.33 | 54.22 |
| Merged (cat) | 58.67 | 56.00 | 49.33 | 54.67 |
| Merged (svd, rank 16) | 58.00 | **56.67** | **50.00** | **54.89** |


## Conclusion

The merging strategy matters more than the merge itself. Naive weight averaging
(`linear`) degrades LoRA adapters because it creates cross-terms between
independently trained factors — on Surgery it performed *worse than no adaptation
at all*. Concatenation-based merging (`cat`, `svd`) avoids this and reaches
joint-training accuracy in seconds instead of ~1 hour of GPU time.

For practitioners: when merging LoRA adapters, `combination_type="svd"` gives the
best trade-off here — joint-training-level accuracy, original adapter size, no
retraining.